# **Taller - Clasificación (Introducción)**

El objetivo del presente taller es realizar una breve introducción a cómo abordar un problema de clasificación. En el próximo taller se verá un caso más elaborado. Por tal motivo, no se hará un procesamiento muy profundo de los datos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

El dataset que usaremos consiste de una serie de mediciones y registros médicos con la intención de predecir si un paciente sufrirá o no un ataque al corazón. El dataset puede ser encontrado en Kaggle: https://www.kaggle.com/fedesoriano/heart-failure-prediction/version/1.

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/LMuroyaT/Lab1/master/datos_taller4.csv')
df.shape

In [ ]:
df.head()

## **Preprocesamiento de Datos**

En este caso, la columna target será HeartDisease. Si la analizamos con más detalle, veremos que ambas clases (0) y (1) están en proporciones casi iguales.

In [ ]:
df['HeartDisease'].value_counts(normalize=True)

Trabajemos primero las columnas numéricas

In [ ]:
df_numericas = df.select_dtypes('number').copy(deep=True)
df_numericas.head()

In [ ]:
# Revisemos valores nulos
df_numericas.isnull().sum()

In [ ]:
# Veamos estadísticos básicos
df_numericas.describe()

In [ ]:
df_numericas.corr()

In [ ]:
# Analicemos correlaciones
# Vemos que fuera de la diagonal principal no hay correlaciones fuertes
import seaborn as sns

df_corr = df_numericas.corr()
sns.heatmap(df_corr, annot=True, cmap='Blues')
plt.show()

Ahora, trabajemos con las variables categoricas

In [ ]:
df_cat = df.select_dtypes('object').copy(deep=True)
df_cat.head()

In [ ]:
# Revisemos valores nulos
df_cat.isnull().sum()

In [ ]:
df_cat.nunique()

In [ ]:
# Notamos que todas las variables son de pocos valores.
# Por ello, podemos binarizarlas usando dummies (One-hot encoding)
# Adicionalmente, podemos usar el truco de eliminar uno de los valores, pues está implícito
df_cat = pd.get_dummies(df_cat, drop_first = True, dtype=int)
df_cat.head()

Ahora, unimos ambos DataFrames para obtener el que se usará para el modelo

In [ ]:
df_numericas.head()

In [ ]:
df_cat.head()

In [ ]:
# Unimos empleando un merge (simil a left join con los índices como llaves)
df_final = df_numericas.merge(df_cat, how = 'left', left_index = True, right_index = True)
df_final.head()

## **Selección de Modelos**

In [ ]:
# Separamos el vector de caracteristicas de la variable respuesta
X = df_final.copy(deep = True)
y = X.pop('HeartDisease')

In [ ]:
from sklearn.model_selection import train_test_split

X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

### **A. Train-validation split**
Vamos a separar el conjunto de datos en entrenamiento (train) y validación (val) para entrenar y medir el modelo respectivamente.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.2, random_state=30, stratify = y_trainval)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
scaler.fit(X_train)
X_train_sc = scaler.transform(X_train)
X_val_sc = scaler.transform(X_val)
X_test_sc = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

lr = LogisticRegression()
dt = DecisionTreeClassifier()
knn5 = KNeighborsClassifier(n_neighbors=5)
knn8 = KNeighborsClassifier(n_neighbors=9)

lr.fit(X_train_sc, y_train)
dt.fit(X_train, y_train)
knn5.fit(X_train_sc, y_train)
knn8.fit(X_train_sc, y_train)

In [ ]:
# Veamos qué predice el clasificador sobre los datos pasados
y_pred_lr = lr.predict(X_val_sc)
y_pred_dt = dt.predict(X_val)
y_pred_knn5 = knn5.predict(X_val_sc)
y_pred_knn8 = knn8.predict(X_val_sc)

In [ ]:
# Podemos predecir las probabilidades de cada categoria
y_proba_lr = lr.predict_proba(X_val_sc)[:,1]
y_proba_dt = dt.predict_proba(X_val)[:,1]
y_proba_knn5 = knn5.predict_proba(X_val_sc)[:,1]
y_proba_knn8 = knn8.predict_proba(X_val_sc)[:,1]

In [ ]:
# Sklearn provee un modulo de metricas que permite hacer el calculo anterior rapidamente
from sklearn.metrics import accuracy_score

print(f"Acc. LR = {accuracy_score(y_val, y_pred_lr)}")
print(f"Acc. DT = {accuracy_score(y_val, y_pred_dt)}")
print(f"Acc. KNN-5 = {accuracy_score(y_val, y_pred_knn5)}")
print(f"Acc. KNN-8 = {accuracy_score(y_val, y_pred_knn8)}")

In [ ]:
# El módulo provee otras métricas de medicion
from sklearn.metrics import precision_score

print(f"Precision LR = {precision_score(y_val, y_pred_lr)}")
print(f"Precision DT = {precision_score(y_val, y_pred_dt)}")
print(f"Precision KNN-5 = {precision_score(y_val, y_pred_knn5)}")
print(f"Precision KNN-8 = {precision_score(y_val, y_pred_knn8)}")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
# Calcular la matriz de confusión
cm = confusion_matrix(y_val, y_pred_lr)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=df["HeartDisease"].unique())
disp.plot(cmap=plt.cm.Blues)
plt.title("Matriz de Confusión - Logistic Regression")
plt.show()

### **B. k-fold Crossvalidation**
Para trabajar correctamente el k-fold crossvalidation requerimos usar Pipelines.

In [ ]:
from sklearn.pipeline import Pipeline

# Creamos un pipeline que escale y luego ejecute el modelo
pipeline_lr = Pipeline([('scaler', MinMaxScaler()), ('lr', LogisticRegression())])
pipeline_dt = Pipeline([('scaler', MinMaxScaler()), ('dt', DecisionTreeClassifier())])
pipeline_knn5 = Pipeline([('scaler', MinMaxScaler()), ('knn5', KNeighborsClassifier(n_neighbors=5))])
pipeline_knn15 = Pipeline([('scaler', MinMaxScaler()), ('knn15', KNeighborsClassifier(n_neighbors=15))])

In [ ]:
from sklearn.model_selection import cross_val_score

scores_lr = cross_val_score(pipeline_lr, X_trainval, y_trainval, cv=5, scoring='accuracy')
scores_dt = cross_val_score(pipeline_dt, X_trainval, y_trainval, cv=5, scoring='accuracy')
scores_knn5 = cross_val_score(pipeline_knn5, X_trainval, y_trainval, cv=5, scoring='accuracy')
scores_knn15 = cross_val_score(pipeline_knn15, X_trainval, y_trainval, cv=5, scoring='accuracy')

print(f"Score LR: {scores_lr.mean()} +/- {scores_lr.std()}")
print(f"Score DT: {scores_dt.mean()} +/- {scores_dt.std()}")
print(f"Score KNN-5: {scores_knn5.mean()} +/- {scores_knn5.std()}")
print(f"Score KNN-15: {scores_knn15.mean()} +/- {scores_knn15.std()}")

## **Optimizacion de hiperparámetros**

A partir del analisis del accuracy, decidimos optimizar los hiperparámetros de LR. Revisando la documentación, dos hiperparámetros de interés son: `C` y `solver`.

In [ ]:
param_grid = {
    'lr__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'lr__solver': ['liblinear', 'saga']
}

In [ ]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(pipeline_lr, param_grid, cv=5, scoring='accuracy')
grid.fit(X_train_sc, y_train)

In [ ]:
grid.best_params_

## **Entrenamiento final del modelo**

In [ ]:
X_dev_sc = scaler.fit_transform(X_trainval)
X_test_sc = scaler.transform(X_test)

lr_final = LogisticRegression(C=10, solver='liblinear')
lr_final.fit(X_dev_sc, y_trainval)

y_pred_lr_final = lr_final.predict(X_test_sc)
print(f"Score en test: {accuracy_score(y_test, y_pred_lr_final)}")